In [2]:
from Bio.PDB import PDBParser, PDBIO, Select

class CleanProtein(Select):
    def accept_residue(self, residue):
        # Keep only standard amino acids (exclude water and ligands)
        if residue.id[0] == ' ':
            return True
        else:
            return False

# Load PDB file
pdb_file = "..\\data\\external\\fold_x\\6m0j.pdb"
parser = PDBParser(QUIET=True)
structure = parser.get_structure("spike", pdb_file)

# Save cleaned structure
io = PDBIO()
io.set_structure(structure)
io.save("6M0J_clean.pdb", CleanProtein())

print("Protein cleaned: water and ligands removed, only standard residues kept.")


Protein cleaned: water and ligands removed, only standard residues kept.


In [ ]:
from pdbfixer import PDBFixer
from openmm.app import PDBFile

fixer = PDBFixer(filename='6M0J_clean.pdb')
fixer.findMissingResidues()
fixer.findMissingAtoms()
fixer.addMissingHydrogens(7.0)  # pH 7.0

PDBFile.writeFile(fixer.topology, fixer.positions, open('6M0J_clean_h.pdb', 'w'))
print("Hydrogens added.")


In [ ]:
import os
import subprocess

input_folder = "data/raw/3D_temp_sdf"
output_folder = "ligands_ready_docking"
os.makedirs(output_folder, exist_ok=True)

for file in os.listdir(input_folder):
    if file.endswith(".sdf"):
        input_path = os.path.join(input_folder, file)
        base_name = os.path.splitext(file)[0]

        output_h_sdf = os.path.join(output_folder, f"{base_name}_3D_h.sdf")
        output_pdbqt = os.path.join(output_folder, f"{base_name}_3D_h.pdbqt")

        # Step 1: Add hydrogens at pH 7.4
        subprocess.run([
            "obabel", "-i", "sdf", input_path,
            "-o", "sdf", "-O", output_h_sdf, "-p", "7.4"
        ], check=True)

        # Step 2: Convert to PDBQT
        subprocess.run([
            "obabel", "-i", "sdf", output_h_sdf,
            "-o", "pdbqt", "-O", output_pdbqt
        ], check=True)

        print(f"Processed {file}: hydrogenated SDF and PDBQT created")


Processed Apilimod(10173277).sdf: hydrogenated SDF and PDBQT created
Processed CamostatMethylate.sdf: hydrogenated SDF and PDBQT created
Processed Cepharanthine(10206).sdf: hydrogenated SDF and PDBQT created
Processed Chloroquine(2719).sdf: hydrogenated SDF and PDBQT created
Processed Darunavir(213039).sdf: hydrogenated SDF and PDBQT created
Processed Ebselen.sdf: hydrogenated SDF and PDBQT created
Processed Favipiravir(492405).sdf: hydrogenated SDF and PDBQT created
Processed HydroxyChloroquine(3652).sdf: hydrogenated SDF and PDBQT created
Processed Ivermectin.sdf: hydrogenated SDF and PDBQT created
Processed Lopinavir.sdf: hydrogenated SDF and PDBQT created
Processed Molnupiravir(145996610).sdf: hydrogenated SDF and PDBQT created
Processed Nelfinavir(64143).sdf: hydrogenated SDF and PDBQT created
Processed Nirmatrelvir(155903259).sdf: hydrogenated SDF and PDBQT created
Processed Nitazoxanide(41684).sdf: hydrogenated SDF and PDBQT created
Processed Remdesivir(121304016).sdf: hydrogena

In [9]:
from rdkit import Chem
from rdkit.Chem import Descriptors, AllChem
from rdkit.Chem import rdMolDescriptors
import pandas as pd
import os

# Folder containing all 17 ligands
ligand_folder = "ligands_ready_docking"
output_csv = "ligand_features_all.csv"

ligand_features = []
ligand_names = []

# Loop over all SDF files in the folder
for sdf_file in os.listdir(ligand_folder):
    if sdf_file.endswith(".sdf"):
        path = os.path.join(ligand_folder, sdf_file)
        suppl = Chem.SDMolSupplier(path)
        
        for mol in suppl:
            if mol is None:
                continue
            
            # Physicochemical descriptors
            mw = Descriptors.MolWt(mol)
            logp = Descriptors.MolLogP(mol)
            h_donors = Descriptors.NumHDonors(mol)
            h_acceptors = Descriptors.NumHAcceptors(mol)
            rot_bonds = Descriptors.NumRotatableBonds(mol)
            tpsa = Descriptors.TPSA(mol)
            
            # Fingerprint (Morgan / ECFP4)
            fp = rdMolDescriptors.GetMorganFingerprintAsBitVect(mol, radius=2, nBits=1024)
            fp_list = list(fp)
            
            # Combine features
            features = [mw, logp, h_donors, h_acceptors, rot_bonds, tpsa] + fp_list
            ligand_features.append(features)
            
            # Ligand name
            name = mol.GetProp("_Name") if mol.HasProp("_Name") else sdf_file.replace(".sdf", "")
            ligand_names.append(name)

# Prepare DataFrame
columns = ["MW","logP","H_donors","H_acceptors","RotatableBonds","TPSA"] + [f"FP_{i}" for i in range(1024)]
df_ligands = pd.DataFrame(ligand_features, columns=columns)
df_ligands["LigandName"] = ligand_names

# Save CSV
df_ligands.to_csv(output_csv, index=False)
print(f"Ligand features extracted for all files. Saved to {output_csv}")


Ligand features extracted for all files. Saved to ligand_features_all.csv


[19:46:52] DEPRECATION WARNING: please use MorganGenerator
[19:46:52] DEPRECATION WARNING: please use MorganGenerator
[19:46:52] DEPRECATION WARNING: please use MorganGenerator
[19:46:52] DEPRECATION WARNING: please use MorganGenerator
[19:46:52] DEPRECATION WARNING: please use MorganGenerator
[19:46:52] DEPRECATION WARNING: please use MorganGenerator
[19:46:52] DEPRECATION WARNING: please use MorganGenerator
[19:46:52] DEPRECATION WARNING: please use MorganGenerator
[19:46:52] DEPRECATION WARNING: please use MorganGenerator
[19:46:52] DEPRECATION WARNING: please use MorganGenerator
[19:46:52] DEPRECATION WARNING: please use MorganGenerator
[19:46:52] DEPRECATION WARNING: please use MorganGenerator
[19:46:52] DEPRECATION WARNING: please use MorganGenerator
[19:46:52] DEPRECATION WARNING: please use MorganGenerator
[19:46:52] DEPRECATION WARNING: please use MorganGenerator
[19:46:52] DEPRECATION WARNING: please use MorganGenerator
[19:46:52] DEPRECATION WARNING: please use MorganGenerat

In [ ]:
from Bio.PDB import PDBParser
import numpy as np
import pandas as pd
import freesasa  # Optional: for SASA calculation, install via pip

# Load protein
parser = PDBParser(QUIET=True)
structure = parser.get_structure("spike", "6m0j_clean_h.pdb")  # cleaned & hydrogenated

# Define pocket residues (example RBD: 331-528)
pocket_residues = [res for res in structure.get_residues() if 331 <= res.id[1] <= 528]

# Residue properties
hydrophobicity_scale = {'A': 1.8,'R': -4.5,'N': -3.5,'D': -3.5,'C': 2.5,'Q': -3.5,
                        'E': -3.5,'G': -0.4,'H': -3.2,'I': 4.5,'L': 3.8,'K': -3.9,
                        'M': 1.9,'F': 2.8,'P': -1.6,'S': -0.8,'T': -0.7,'W': -0.9,
                        'Y': -1.3,'V': 4.2}

polar_residues = {'R','K','H','D','E','N','Q','S','T','Y'}
positive_residues = {'R','K','H'}
negative_residues = {'D','E'}
aromatic_residues = {'F','W','Y','H'}

# Compute residue-level features
hydrophobicity = [hydrophobicity_scale.get(res.get_resname()[0], 0) for res in pocket_residues]
avg_hydro = np.mean(hydrophobicity)
std_hydro = np.std(hydrophobicity)

num_polar = sum(1 for res in pocket_residues if res.get_resname()[0] in polar_residues)
frac_polar = num_polar / len(pocket_residues)

num_positive = sum(1 for res in pocket_residues if res.get_resname()[0] in positive_residues)
num_negative = sum(1 for res in pocket_residues if res.get_resname()[0] in negative_residues)
frac_charge = (num_positive + num_negative) / len(pocket_residues)

num_aromatic = sum(1 for res in pocket_residues if res.get_resname()[0] in aromatic_residues)
frac_aromatic = num_aromatic / len(pocket_residues)

pocket_size = len(pocket_residues)

# Optional: compute SASA using freesasa
try:
    structure_str = "6m0j_clean_h.pdb"
    result = freesasa.Structure(structure_str)
    sasa_result = freesasa.calc(result)
    sasa = sasa_result.totalArea()
except:
    sasa = np.nan

# Combine features
protein_features = [avg_hydro, std_hydro, frac_polar, frac_charge, frac_aromatic, pocket_size, sasa]

# Save to CSV
columns = ["AvgHydro","StdHydro","FracPolar","FracCharge","FracAromatic","PocketSize","SASA"]
df_protein = pd.DataFrame([protein_features], columns=columns)
df_protein.to_csv("protein_features_rich.csv", index=False)

print("✅ Protein pocket features extracted (rich set).")


In [15]:
import pandas as pd

# Path to your file
csv_path = "../data/processed/final_dataset_ready2.csv"

# Load dataset
df = pd.read_csv(csv_path)

# Check if the column exists
if "mutation" not in df.columns:
    raise ValueError("The CSV does not contain a 'mutation' column.")

# Get unique mutations as a list
mutations = df["mutation"].dropna().unique().tolist()

# Save to a text file (one mutation per line)
with open("../data/processed/mutation_list.txt", "w") as f:
    for m in mutations:
        f.write(str(m) + "\n")

# Alternatively, save to a CSV
pd.DataFrame(mutations, columns=["mutation"]).to_csv(
    "../data/processed/mutation_list.csv", index=False
)

print("Saved mutation list to mutation_list.txt and mutation_list.csv")


Saved mutation list to mutation_list.txt and mutation_list.csv


In [ ]:
import pandas as pd
import numpy as np

# -----------------------------
# 1️⃣ Load ligand features
# -----------------------------
df_ligands = pd.read_csv("ligand_features_all.csv")

# -----------------------------
# 2️⃣ Define pocket sequence
# Use 1-letter codes for residues 331-528
# -----------------------------
pocket_residues = ['N', 'I', 'T', 'N', 'L', 'C', 'P', 'F', 'G', 'E', 'V', 'F', 'N', 'A', 
                   'T', 'R', 'F', 'A', 'S', 'V', 'Y', 'A', 'W', 'N', 'R', 'K', 'R', 'I', 
                   'S', 'N', 'C', 'V', 'A', 'D', 'Y', 'S', 'V', 'L', 'Y', 'N', 'S', 'A', 
                   'S', 'F', 'S', 'T', 'F', 'K', 'C', 'Y', 'G', 'V', 'S', 'P', 'T', 'K', 
                   'L', 'N', 'D', 'L', 'C', 'F', 'T', 'N', 'V', 'Y', 'A', 'D', 'S', 'F', 
                   'V', 'I', 'R', 'G', 'D', 'E', 'V', 'R', 'Q', 'I', 'A', 'P', 'G', 'Q', 
                   'T', 'G', 'K', 'I', 'A', 'D', 'Y', 'S', 'Y', 'K', 'L', 'P', 'D', 'D', 
                   'F', 'T', 'G', 'C', 'V', 'I', 'A', 'W', 'N', 'S', 'N', 'N', 'L', 'D', 
                   'S', 'K', 'V', 'G', 'G', 'N', 'Y', 'N', 'Y', 'L', 'Y', 'R', 'L', 'F', 
                   'R', 'K', 'S', 'N', 'L', 'K', 'P', 'F', 'E', 'R', 'D', 'I', 'S', 'T', 
                   'E', 'I', 'Y', 'Q', 'A', 'G', 'S', 'T', 'P', 'C', 'N', 'G', 'V', 'E', 
                   'G', 'F', 'N', 'C', 'Y', 'F', 'P', 'L', 'Q', 'S', 'Y', 'G', 'F', 'Q', 
                   'P', 'T', 'N', 'G', 'V', 'G', 'Y', 'Q', 'P', 'Y', 'R', 'V', 'V', 'V', 
                   'L', 'S', 'F', 'E', 'L', 'L', 'H', 'A', 'P', 'A', 'T', 'V', 'C', 'G', 
                   'P', 'K', 'K', 'S', 'T'] 

# -----------------------------
# 3️⃣ Load mutation list from file
# -----------------------------
# Option A: if you saved as CSV
df_mut = pd.read_csv("../data/processed/mutation_list.csv")
mutations = df_mut["mutation"].dropna().tolist()

# Option B: if you saved as TXT
# with open("data/processed/mutation_list.txt") as f:
#     mutations = [line.strip() for line in f if line.strip()]

print("Loaded", len(mutations), "mutations.")

# -----------------------------
# 4️⃣ Residue property dictionaries
# -----------------------------
hydrophobicity_scale = {'A': 1.8,'R': -4.5,'N': -3.5,'D': -3.5,'C': 2.5,'Q': -3.5,
                        'E': -3.5,'G': -0.4,'H': -3.2,'I': 4.5,'L': 3.8,'K': -3.9,
                        'M': 1.9,'F': 2.8,'P': -1.6,'S': -0.8,'T': -0.7,'W': -0.9,
                        'Y': -1.3,'V': 4.2}

polar_residues = {'R','K','H','D','E','N','Q','S','T','Y'}
positive_residues = {'R','K','H'}
negative_residues = {'D','E'}
aromatic_residues = {'F','W','Y','H'}

# -----------------------------
# 5️⃣ Function to compute protein features for a mutation
# -----------------------------
def mutate_protein_features(pocket_residues, mutation):
    wt_res = mutation[0]
    mut_res = mutation[-1]
    pos = int(mutation[1:-1])
    
    mutated_pocket = pocket_residues.copy()
    idx = pos - 331  # assuming pocket_residues starts at 331
    if 0 <= idx < len(mutated_pocket):
        mutated_pocket[idx] = mut_res
    
    hydro = [hydrophobicity_scale.get(res, 0) for res in mutated_pocket]
    avg_hydro = np.mean(hydro)
    std_hydro = np.std(hydro)
    
    num_polar = sum(1 for res in mutated_pocket if res in polar_residues)
    frac_polar = num_polar / len(mutated_pocket)
    
    num_positive = sum(1 for res in mutated_pocket if res in positive_residues)
    num_negative = sum(1 for res in mutated_pocket if res in negative_residues)
    frac_charge = (num_positive + num_negative) / len(mutated_pocket)
    
    num_aromatic = sum(1 for res in mutated_pocket if res in aromatic_residues)
    frac_aromatic = num_aromatic / len(mutated_pocket)
    
    pocket_size = len(mutated_pocket)
    
    return [avg_hydro, std_hydro, frac_polar, frac_charge, frac_aromatic, pocket_size]

# -----------------------------
# 6️⃣ Generate dataset
# -----------------------------
rows = []
for mut in mutations:
    protein_vector = mutate_protein_features(pocket_residues, mut)
    for _, ligand_row in df_ligands.iterrows():
        row = list(ligand_row.drop("LigandName")) + protein_vector + [ligand_row["LigandName"], mut]
        rows.append(row)

# -----------------------------
# 7️⃣ Create DataFrame & save CSV
# -----------------------------
feature_columns = list(df_ligands.drop("LigandName", axis=1).columns) + \
                  ["AvgHydro","StdHydro","FracPolar","FracCharge","FracAromatic","PocketSize"] + \
                  ["LigandName","Mutation"]

df_all = pd.DataFrame(rows, columns=feature_columns)
df_all.to_csv("ligand_mutation_dataset.csv", index=False)

print("✅ Ligand-mutation dataset CSV created. Rows:", len(df_all))


In [ ]:
import pandas as pd
import numpy as np
import re

# -----------------------------
# 1️⃣ Load ligand features
# -----------------------------
df_ligands = pd.read_csv("ligand_features_all.csv")

# -----------------------------
# 2️⃣ Define pocket sequence
# -----------------------------
pocket_residues = ['N','I','T','N','L','C','P','F','G','E','V','F','N','A',
                   'T','R','F','A','S','V','Y','A','W','N','R','K','R','I',
                   'S','N','C','V','A','D','Y','S','V','L','Y','N','S','A',
                   'S','F','S','T','F','K','C','Y','G','V','S','P','T','K',
                   'L','N','D','L','C','F','T','N','V','Y','A','D','S','F',
                   'V','I','R','G','D','E','V','R','Q','I','A','P','G','Q',
                   'T','G','K','I','A','D','Y','S','Y','K','L','P','D','D',
                   'F','T','G','C','V','I','A','W','N','S','N','N','L','D',
                   'S','K','V','G','G','N','Y','N','Y','L','Y','R','L','F',
                   'R','K','S','N','L','K','P','F','E','R','D','I','S','T',
                   'E','I','Y','Q','A','G','S','T','P','C','N','G','V','E',
                   'G','F','N','C','Y','F','P','L','Q','S','Y','G','F','Q',
                   'P','T','N','G','V','G','Y','Q','P','Y','R','V','V','V',
                   'L','S','F','E','L','L','H','A','P','A','T','V','C','G',
                   'P','K','K','S','T'] 

# -----------------------------
# 3️⃣ Load mutation list
# -----------------------------
df_mut = pd.read_csv("../data/processed/mutation_list.csv")
raw_mutations = df_mut["mutation"].dropna().tolist()

# -----------------------------
# 4️⃣ Validate mutation format
# -----------------------------
valid_mutations = []
pattern = re.compile(r"^[A-Z]\d+[A-Z]$")

for mut in raw_mutations:
    if not pattern.match(mut):
        print(f"⚠️ Skipping invalid mutation format: {mut}")
        continue
    wt_res, mut_res = mut[0], mut[-1]
    pos = int(mut[1:-1])
    idx = pos - 331
    if idx < 0 or idx >= len(pocket_residues):
        print(f"⚠️ Skipping out-of-range mutation: {mut}")
        continue
    valid_mutations.append(mut)

print(f"✅ Loaded {len(valid_mutations)} valid mutations out of {len(raw_mutations)}")

# -----------------------------
# 5️⃣ Dictionaries
# -----------------------------
hydrophobicity_scale = {'A': 1.8,'R': -4.5,'N': -3.5,'D': -3.5,'C': 2.5,'Q': -3.5,
                        'E': -3.5,'G': -0.4,'H': -3.2,'I': 4.5,'L': 3.8,'K': -3.9,
                        'M': 1.9,'F': 2.8,'P': -1.6,'S': -0.8,'T': -0.7,'W': -0.9,
                        'Y': -1.3,'V': 4.2}

polar_residues = {'R','K','H','D','E','N','Q','S','T','Y'}
positive_residues = {'R','K','H'}
negative_residues = {'D','E'}
aromatic_residues = {'F','W','Y','H'}

# -----------------------------
# 6️⃣ Mutation feature function
# -----------------------------
def mutate_protein_features(pocket_residues, mutation):
    wt_res = mutation[0]
    mut_res = mutation[-1]
    pos = int(mutation[1:-1])
    
    mutated_pocket = pocket_residues.copy()
    idx = pos - 331
    mutated_pocket[idx] = mut_res
    
    hydro = [hydrophobicity_scale.get(res, 0) for res in mutated_pocket]
    avg_hydro = np.mean(hydro)
    std_hydro = np.std(hydro)
    
    num_polar = sum(1 for res in mutated_pocket if res in polar_residues)
    frac_polar = num_polar / len(mutated_pocket)
    
    num_positive = sum(1 for res in mutated_pocket if res in positive_residues)
    num_negative = sum(1 for res in mutated_pocket if res in negative_residues)
    frac_charge = (num_positive + num_negative) / len(mutated_pocket)
    
    num_aromatic = sum(1 for res in mutated_pocket if res in aromatic_residues)
    frac_aromatic = num_aromatic / len(mutated_pocket)
    
    pocket_size = len(mutated_pocket)
    
    return [avg_hydro, std_hydro, frac_polar, frac_charge, frac_aromatic, pocket_size]

# -----------------------------
# 7️⃣ Generate dataset
# -----------------------------
rows = []
for mut in valid_mutations:
    protein_vector = mutate_protein_features(pocket_residues, mut)
    for _, ligand_row in df_ligands.iterrows():
        row = list(ligand_row.drop("LigandName")) + protein_vector + [ligand_row["LigandName"], mut]
        rows.append(row)

# -----------------------------
# 8️⃣ Create DataFrame & save
# -----------------------------
feature_columns = list(df_ligands.drop("LigandName", axis=1).columns) + \
                  ["AvgHydro","StdHydro","FracPolar","FracCharge","FracAromatic","PocketSize"] + \
                  ["LigandName","Mutation"]

df_all = pd.DataFrame(rows, columns=feature_columns)
df_all.to_csv("ligand_mutation_dataset2.csv", index=False)

print("Ligand-mutation dataset created. Rows:", len(df_all))


In [4]:
import pandas as pd 
df = pd.read_csv("ligand_mutation_dataset.csv")
print(df.shape)

(64923, 1038)
